In [12]:
import pandas as pd
from scipy import stats
import numpy as np

In [13]:
df = pd.read_csv("ab_data.csv")
df.head()

,user_id,timestamp,group,landing_page,converted
0,851104,2017-01-21 22:11:48.556739,control,old_page,0
1,804228,2017-01-12 08:01:45.159739,control,old_page,0
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
4,864975,2017-01-21 01:52:26.210827,control,old_page,1


In [15]:
print(df.shape)
print(df.isnull().sum())

(294478, 5)
user_id         0
timestamp       0
group           0
landing_page    0
converted       0
dtype: int64


In [17]:
df.dtypes

user_id          int64
timestamp       object
group           object
landing_page    object
converted        int64
dtype: object

In [16]:
print(df.duplicated(subset='user_id').sum())

3894


In [ ]:
dupe_ids = df[df.duplicated(subset='user_id', keep=False)]
print(dupe_ids.sort_values('user_id').head(10))

        user_id                   timestamp      group landing_page  converted
213114   630052  2017-01-07 12:25:54.089486  treatment     old_page          1
230259   630052  2017-01-17 01:16:05.208766  treatment     new_page          0
22513    630126  2017-01-14 13:35:54.778695  treatment     old_page          0
251762   630126  2017-01-19 17:16:00.280440  treatment     new_page          0
11792    630137  2017-01-22 14:59:22.051308    control     new_page          0
183371   630137  2017-01-20 02:08:49.893878    control     old_page          0
207211   630320  2017-01-07 18:02:43.626318    control     old_page          0
255753   630320  2017-01-12 05:27:37.181803  treatment     old_page          0
96929    630471  2017-01-07 02:14:17.405726    control     new_page          0
110634   630471  2017-01-23 01:42:51.501851    control     old_page          0


In [19]:
mismatch = df[((df.group == 'control') & (df.landing_page == 'new_page')) |
              ((df.group == 'treatment') & (df.landing_page == 'old_page'))]
print("Mismatched rows:", len(mismatch))

Mismatched rows: 3893


In [20]:
# Step 1: remove all mismatched rows
df_clean = df.drop(mismatch.index)

# Step 2: check if any duplicates remain after removing mismatches
remaining_dupes = df_clean.duplicated(subset='user_id').sum()
print("Remaining duplicate user_ids after removing mismatches:", remaining_dupes)

print("Shape before:", df.shape)
print("Shape after removing mismatches:", df_clean.shape)

Remaining duplicate user_ids after removing mismatches: 1
Shape before: (294478, 5)
Shape after removing mismatches: (290585, 5)


In [22]:
leftover = df_clean[df_clean.duplicated(subset='user_id', keep=False)]
print(leftover)

      user_id                   timestamp      group landing_page  converted
1899   773192  2017-01-09 05:37:58.781806  treatment     new_page          0
2893   773192  2017-01-14 02:55:59.590927  treatment     new_page          0


In [23]:
df_clean = df_clean.drop_duplicates(subset='user_id', keep='first')
print("Final shape:", df_clean.shape)

Final shape: (290584, 5)


In [24]:
summary = df_clean.groupby('group')['converted'].agg(['count', 'sum', 'mean'])
print(summary)

            count    sum      mean
group                             
control    145274  17489  0.120386
treatment  145310  17264  0.118808


In [26]:
n_con = summary.loc['control', 'count']
n_treat = summary.loc['treatment', 'count']
conv_con = summary.loc['control', 'sum']
conv_treat = summary.loc['treatment', 'sum']

In [28]:
p_con = conv_con / n_con
p_treat = conv_treat / n_treat
p_pool = (conv_con + conv_treat) / (n_con + n_treat)

Why bother merging them? Because the test works by temporarily pretending the null hypothesis is true — pretending the old page and new page really convert at the exact same rate, and any difference we saw was just random luck. If that pretend-scenario were true, there'd be no real "control rate" vs "treatment rate" — there'd just be one single true conversion rate shared by both. p_pool is our best estimate of what that single shared rate would be, using all 290,584 users' data combined.

In [29]:
se = np.sqrt(p_pool * (1 - p_pool) * (1/n_con + 1/n_treat))
z = (p_treat - p_con) / se
p_value = 2 * (1 - stats.norm.cdf(abs(z)))

print(f"Control rate: {p_con:.4f}")
print(f"Treatment rate: {p_treat:.4f}")
print(f"Z-score: {z:.4f}")
print(f"P-value: {p_value:.4f}")

Control rate: 0.1204
Treatment rate: 0.1188
Z-score: -1.3109
P-value: 0.1899
